# Create Sync Tables from Unity Catalog to Lakebase

This notebook creates synced tables to automatically synchronize Delta tables to Lakebase.


In [ ]:
# Step 1: Install SDK and Setup
%pip install databricks-sdk --upgrade --quiet
dbutils.library.restartPython()


In [ ]:
# Step 2: Configuration (re-define after restart)
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.database import SyncedDatabaseTable, SyncedTableSpec, NewPipelineSpec, SyncedTableSchedulingPolicy

w = WorkspaceClient()

# Unity Catalog source
CATALOG = "lakemeter_catalog"
SCHEMA = "lakemeter"

# Lakebase target
LAKEBASE_INSTANCE = "lakemeter-db"
LAKEBASE_DATABASE = "lakemeter_pricing"

# Pipeline storage
STORAGE_CATALOG = CATALOG
STORAGE_SCHEMA = SCHEMA

print(f"Source: {CATALOG}.{SCHEMA}")
print(f"Target: {LAKEBASE_INSTANCE}.{LAKEBASE_DATABASE}")


In [ ]:
# Step 3: Enable Change Data Feed on source tables
print("="*70)
print("Step 3: Enable Change Data Feed")
print("="*70)

source_tables = [
    "dbu_prices", "vm_costs", "dbsql_rates", "serverless_product_rates",
    "fmapi_databricks_rates", "fmapi_proprietary_rates", "sku_region_mapping",
    "instance_rates", "dbu_multipliers", "dbsql_warehouse_config"
]

for table in source_tables:
    full_name = f"{CATALOG}.{SCHEMA}.{table}"
    try:
        spark.sql(f"ALTER TABLE {full_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
        print(f"✅ {table}")
    except Exception as e:
        print(f"⚠️ {table}: {str(e)[:50]}")


In [ ]:
# Step 4: Tables to sync with their PKs
TABLES_TO_SYNC = [
    {"source": "dbu_prices", "target": "pricing_dbu_rates", "pk": ["cloud", "region", "tier", "product_type", "sku_name"]},
    {"source": "vm_costs", "target": "pricing_vm_costs", "pk": ["cloud", "region", "instance_type", "pricing_tier", "payment_option"]},
    {"source": "dbsql_rates", "target": "product_dbsql_rates", "pk": ["cloud", "warehouse_type", "warehouse_size"]},
    {"source": "serverless_product_rates", "target": "product_serverless_rates", "pk": ["cloud", "product", "size_or_model"]},
    {"source": "fmapi_databricks_rates", "target": "product_fmapi_databricks", "pk": ["model", "rate_type"]},
    {"source": "fmapi_proprietary_rates", "target": "product_fmapi_proprietary", "pk": ["cloud", "provider", "model", "rate_type", "endpoint_type", "context_length"]},
    {"source": "sku_region_mapping", "target": "ref_sku_region_map", "pk": ["cloud", "sku_region"]},
    {"source": "instance_rates", "target": "ref_instance_dbu_rates", "pk": ["cloud", "instance_type"]},
    {"source": "dbu_multipliers", "target": "ref_dbu_multipliers", "pk": ["sku_type", "feature"]},
    {"source": "dbsql_warehouse_config", "target": "ref_dbsql_warehouse_config", "pk": ["cloud", "warehouse_type", "warehouse_size"]},
]

print(f"Tables to sync: {len(TABLES_TO_SYNC)}")
for t in TABLES_TO_SYNC:
    print(f"  {t['source']} -> {t['target']} (PK: {t['pk']})")


In [ ]:
# Step 5: Create Synced Tables
print("="*70)
print("Step 5: Creating Synced Tables")
print("="*70)

results = {"created": [], "exists": [], "failed": []}

for table in TABLES_TO_SYNC:
    synced_table_name = f"{CATALOG}.{SCHEMA}.sync_{table['target']}"
    source_table = f"{CATALOG}.{SCHEMA}.{table['source']}"
    
    print(f"\n{table['source']} -> {table['target']}:")
    
    try:
        synced_table = w.database.create_synced_database_table(
            SyncedDatabaseTable(
                name=synced_table_name,
                database_instance_name=LAKEBASE_INSTANCE,
                logical_database_name=LAKEBASE_DATABASE,
                spec=SyncedTableSpec(
                    source_table_full_name=source_table,
                    primary_key_columns=table['pk'],
                    scheduling_policy=SyncedTableSchedulingPolicy.TRIGGERED,
                    create_database_objects_if_missing=True,
                    new_pipeline_spec=NewPipelineSpec(
                        storage_catalog=STORAGE_CATALOG,
                        storage_schema=STORAGE_SCHEMA
                    )
                ),
            )
        )
        print(f"   ✅ Created")
        results["created"].append(table['target'])
    except Exception as e:
        err = str(e)
        if "already exists" in err.lower():
            print(f"   ⏭️ Already exists")
            results["exists"].append(table['target'])
        else:
            print(f"   ❌ {err[:80]}")
            results["failed"].append(table['target'])

print("\n" + "="*70)
print(f"Created: {len(results['created'])}, Exists: {len(results['exists'])}, Failed: {len(results['failed'])}")
if results['failed']:
    print(f"Failed: {results['failed']}")


In [ ]:
# Step 6: Trigger Sync for All Tables
print("="*70)
print("Step 6: Trigger Sync for All Tables")
print("="*70)

for table in TABLES_TO_SYNC:
    synced_table_name = f"{CATALOG}.{SCHEMA}.sync_{table['target']}"
    try:
        resp = w.database.get_synced_database_table(synced_table_name)
        pipeline_id = resp.data_synchronization_status.pipeline_id
        w.pipelines.start_update(pipeline_id=pipeline_id)
        print(f"✅ {table['target']} - sync triggered")
    except Exception as e:
        print(f"❌ {table['target']} - {str(e)[:50]}")


In [ ]:
# Step 7: Check Sync Status
import time
print("="*70)
print("Step 7: Sync Table Status")
print("="*70)

print("Waiting 30 seconds for sync to start...")
time.sleep(30)

online = 0
pending = 0
failed = 0

for table in TABLES_TO_SYNC:
    synced_table_name = f"{CATALOG}.{SCHEMA}.sync_{table['target']}"
    try:
        info = w.database.get_synced_database_table(synced_table_name)
        state = info.as_dict().get('data_synchronization_status', {}).get('state', 'UNKNOWN')
        
        if 'ONLINE' in str(state):
            print(f"✅ {table['target']}: {state}")
            online += 1
        elif 'PENDING' in str(state) or 'PROVISIONING' in str(state):
            print(f"⏳ {table['target']}: {state}")
            pending += 1
        else:
            print(f"❌ {table['target']}: {state}")
            failed += 1
    except Exception as e:
        print(f"❌ {table['target']}: Not found")
        failed += 1

print(f"\n📊 Summary: {online} online, {pending} pending, {failed} failed")
